# Day 08 · 하네스 — 에이전트의 작업 환경

7강까지는 **손으로** 에이전트를 만들었다. 오늘부터는 **실제 코드 에이전트(Claude Code · Codex)를 켜고** 그 작업 환경을 세팅한다.

> 같은 모델도 **하네스가 결과를 가른다.**

**오늘의 실습 3개** — 모두 터미널·Claude Code에서 진행한다 (파이썬 실행 셀 없음).

| 랩 | 무엇을 | 결과물 |
|---|---|---|
| Lab 1 | buggy-request 레포로 **CLAUDE.md 작성** | 내 프로젝트 룰 파일 |
| Lab 2 | 같은 레포에서 **심어둔 결함 찾기** | 회수율 측정표 |
| Lab 3 | **슬랙을 MCP로 연결** | 말 한마디로 슬랙 게시 |

**사전 준비** — Claude Code 설치·로그인 (Pro/Max/Team/Console 플랜 필요, 무료 플랜 불가). 설치는 `day08/Claude_Code_Codex_세팅.md` 참고.


## Lab 1 · buggy-request 레포로 CLAUDE.md 작성

완벽한 룰 파일이 목표가 아니다. **초안을 빠르게 뽑고, 실수마다 한 줄씩 추가**하는 게 정석이다.

### 1. 클론 → /init

```bash
git clone https://github.com/TunaLee/buggy-request.git
cd buggy-request
claude
```

```
> /init
  코드베이스 분석 → CLAUDE.md 초안 자동 생성
```

### 2. /memory 로 로드 확인

```
> /memory
  로드된 파일 목록 · 자동 메모리 ON/OFF
```

목록에 CLAUDE.md가 **없으면** 위치·파일명 문제다 — Claude가 아예 못 본 것.

### 3. 5섹션으로 정리

```markdown
# 스택
Next.js 15 (App Router) · TypeScript · Prisma + Postgres

# 디렉토리
src/app/   페이지·API
prisma/    스키마·migration
tests/     vitest 단위·통합

# 명령어
npm run dev · npm test · npx prisma migrate dev

# 룰
- server component 우선 (use client 최소화)
- import 절대경로 @/...
- 새 API는 zod로 입력 검증

# 하지 말 것
- .env / .env.local 커밋
- any 사용
- 테스트 없이 라우트 추가
```

### 4. 같은 태스크를 전/후로 비교

`"테스트 1개 추가해줘"` 를 **CLAUDE.md 없이 / 있이** 각각 시켜본다. 무엇이 달라지는가?

### 5. 빗나간 부분 → 룰 한 줄로 환원

- `"any 썼네"` → **하지 말 것**에 추가
- `"테스트 위치 틀렸네"` → **디렉토리**에 추가

### 막혔을 때

한 줄도 안 떠오르면 — **최근 에이전트가 가장 짜증나게 했던 실수 1개**만 떠올려 "하지 말 것"에 적는다. 거기서부터 거꾸로 채워진다.

### 원리

CLAUDE.md는 **강제가 아니라 컨텍스트**다. 따르려고 읽는 것일 뿐, 반드시 막아야 하면 **Hook**으로 (9강).

마지막에 `git commit` — 다음 세션부터 **프롬프트 캐싱**이 걸린다.


## Lab 2 · 심어둔 결함 찾기 (회수율 측정)

Lab 1에서 클론한 **그 buggy-request 레포 그대로**. 곳곳에 결함이 숨어 있다 — 보안 · 로직 · 런타임.

작성한 CLAUDE.md를 컨텍스트로, **어떤 도구 조합이 결함을 얼마나 잡아내는지 직접 측정**한다.

### 측정 절차 — 변수를 하나씩만 바꾼다

| 단계 | 명령 | 찾은 결함 수 |
|---|---|---|
| ① | `/review` 단독 | ___ 개 |
| ② | `+ /cso` (보안 특화) | ___ 개 |
| ③ | `+ 테스트 · /qa` (런타임) | ___ 개 |

```
> /review
  ... 발견 항목 기록

> /cso
  보안 특화 점검 — 회수율 변화 확인

> /qa
  런타임에서만 드러나는 결함 (off-by-one · 상태 전이 · 삭제 미반영)
```

### 왜 이렇게 하나

어떤 도구·조합이 회수율을 끌어올리는지는 **감이 아니라 숫자로** 확인한다. 변수를 하나씩만 바꿔 재측정하면 차이가 드러난다.

### 리스크 티어 → 명령 매핑

| 리스크 | 명령 |
|---|---|
| 낮음 | `/code-review` · `/qa` |
| 중간 | + `/security-review` · `/review` |
| 높음 | + `/cso` · **사람 최종 승인** |

### 원리

에이전트는 **제안이 아니라 실행**한다. 삼성(데이터 유출) · EchoLeak(프롬프트 인젝션) · Replit(폭주) — 셋 다 나중에 막을 문제가 아니라 **설계에서 막을 문제**였다.

> 이 회수율 측정 방식이 곧 **도입 보고서의 근거**가 된다 (9강 도입 로드맵).


## Lab 3 · 슬랙을 MCP로 연결

사내 시스템을 에이전트에 잇는다. **노출한 도구가 곧 보안 경계**다.

### 1. Slack 앱 생성

`api.slack.com` → **Create an App** → From scratch (앱 이름 · 워크스페이스 선택)

### 2. OAuth 스코프 추가

**OAuth & Permissions** → Bot Token Scopes 에 `chat:write` 등 추가

### 3. 워크스페이스에 설치 → Allow

스코프를 넣으면 **Install to Workspace**가 활성화된다. 허용하면 토큰 발급.

### 4. 두 값 복사

- **Bot User OAuth Token** (`xoxb-…`) → `SLACK_BOT_TOKEN` — **비밀값이다. 마스킹·외부 노출 금지**
- 주소창 URL의 첫 ID (`T…`) → `SLACK_TEAM_ID`

### 5. MCP 등록

```bash
claude mcp add slack \
  -e SLACK_BOT_TOKEN=xoxb-… \
  -e SLACK_TEAM_ID=T… \
  -- npx -y @modelcontextprotocol/server-slack
```

재시작 후 `/mcp` 로 `slack connected` 확인. 봇을 채널에 `/invite` 한다.

### 6. 말 한마디로 게시

```
> #dev 채널에 오늘 변경사항 요약해서 올려줘
  slack.post_message
```

### 왜 봇 토큰인가

공식 서버(`mcp.slack.com`)는 Claude Code의 자동 인증(DCR)을 받지 않아 **인증에 실패한다**. 그래서 봇 토큰으로 붙인다.

### 트러블슈팅 — `-32000`

서버가 떴다 **즉시 종료**됐다는 뜻(Connection closed). 원인은 거의 **env · 패키지 · 윈도우 spawn**이다.

```bash
# 1. 진짜 에러 보기 — Claude는 stderr를 숨긴다
SLACK_BOT_TOKEN=xoxb-… SLACK_TEAM_ID=T… npx -y @modelcontextprotocol/server-slack
claude --debug

# 2. 등록값 확인 — 토큰·TeamID 오타가 가장 흔하다
claude mcp get slack

# 3. Windows — cmd /c 로 감싸 stdio 즉시 종료 방지
claude mcp add slack -e … -- cmd /c npx -y @modelcontextprotocol/server-slack
```

### 원리

MCP = **웨이터**다. 머리는 Claude, 손발은 MCP 서버. **메뉴에 없는 건 못 시킨다** — 노출한 도구만 호출되니 그게 곧 보안 경계다.

> 5강에서 만든 우리 MCP 서버도 **똑같은 방식**으로 붙는다: `claude mcp add day07-tools -- python server.py`


## 산출물 체크리스트

- [ ] buggy-request 레포에서 `/init` 으로 **CLAUDE.md 초안**을 뽑았다
- [ ] `/memory` 로 **로드되는지 확인**했다
- [ ] 5섹션(스택·디렉토리·명령어·룰·하지 말 것)으로 정리했다
- [ ] 같은 태스크를 **CLAUDE.md 전/후로 비교**했다
- [ ] 빗나간 부분을 **룰 한 줄로 환원**했다
- [ ] `/review` → `+/cso` → `+/qa` 로 **회수율을 3번 측정**해 표로 남겼다
- [ ] Slack 앱을 만들어 **MCP로 등록**했다
- [ ] **말 한마디로 슬랙에 게시**했다
- [ ] 토큰 절감 레버(입력 줄이기 · 모델 라우팅)를 **내 작업에 하나 적용**해봤다

> 한 줄: **모델은 그대로다. 하네스가 결과를 가른다.**
